# Ark+ (Swin-L) -- abstract improvement test: class-weighted BCE

Cells 1-14 below are the **unmodified baseline** setup + plain-BCE Ark+ training cell, copied
verbatim from `ArkPlus_Colab_Baseline.ipynb` / `ArkPlus_Gaps_Colab.ipynb` -- run them first (or
skip straight to the new cells below if the plain-baseline screening result is already cached
on Drive from a previous run, since all three notebooks share the same `DRIVE_CKPT_DIR`).

New cells at the end run a quick, proof-of-viability test of **class-weighted BCE** -- one of
the 5 planned interventions listed in the SPIE supplemental PDF, targeting the Atelectasis/
Consolidation AP/F1 weakness found in the real Ark+ headline result. Screening tier only
(1 seed, 20% subset, still the full 30 epochs) for speed ahead of the Aug 5 deadline -- this is
directional screening-tier evidence, not a headline-tier confirmed result.

In [ ]:
!nvidia-smi --query-gpu=name,memory.total,driver_version --format=csv

In [ ]:
# Colab's pre-installed stack (pandas, etc.) is compiled against numpy>=2's actual C ABI --
# this isn't just a pip-metadata preference, running them against numpy<2 crashes for real
# ("numpy.dtype size changed, may indicate binary incompatibility"). We previously tried
# downgrading numpy to dodge an *unrelated* numpy-2.0 incompatibility in imgaug (a transitive
# dep of albumentations) -- that broke pandas instead, and the downgrade didn't even take
# effect this-session anyway, since Colab pre-imports numpy before any of our cells run. So:
# make sure numpy is restored to >=2 here (undoing any earlier accidental downgrade left on
# disk), and fix the imgaug incompatibility directly instead (see section 2, patches 4 & 5).
import sys
!{sys.executable} -m pip install -q --break-system-packages "numpy>=2"
import numpy
print('numpy', numpy.__version__, '@', numpy.__file__)
print('kernel python:', sys.executable)

## 1. Kaggle credentials + dataset download

In [ ]:
import os, json, getpass, sys

kaggle_username = input('Kaggle username: ')
kaggle_key = getpass.getpass('Kaggle API key: ')

os.makedirs('/root/.kaggle', exist_ok=True)
with open('/root/.kaggle/kaggle.json', 'w') as f:
    json.dump({"username": kaggle_username, "key": kaggle_key}, f)
os.chmod('/root/.kaggle/kaggle.json', 0o600)
del kaggle_key

!{sys.executable} -m pip install -q --break-system-packages kaggle
os.makedirs('/content/data', exist_ok=True)

# Using the kaggle package's own Python API instead of any CLI form -- it turns out this
# version of the package has no `__main__.py` (`-m kaggle` fails with "cannot be directly
# executed"), and the bare `kaggle` console script hits the same PATH-mismatch class of
# issue we saw with pip earlier. Calling the API in-process sidesteps all of that.
import kaggle
kaggle.api.authenticate()
# quiet=False -- dataset_download_files defaults to quiet=True, which is why this ran with
# zero output before. With it off, kaggle's own tqdm byte-progress bar (MB downloaded, speed,
# ETA) shows up instead of the download appearing to hang silently for several minutes.
kaggle.api.dataset_download_files('ashery/chexpert', path='/content/data', unzip=True, quiet=False)
print('Download complete.')

In [ ]:
# The train.csv/valid.csv paths are prefixed with 'CheXpert-v1.0-small/', but the
# Kaggle extract puts train/ and valid/ directly under /content/data. Symlink to bridge it.
import os
data_root = '/content/data'
link = os.path.join(data_root, 'CheXpert-v1.0-small')
if not os.path.islink(link) and not os.path.exists(link):
    os.symlink(data_root, link)

print(sorted(os.listdir(data_root)))
import pandas as pd
df = pd.read_csv(os.path.join(data_root, 'train.csv'))
print(df.shape, list(df.columns))

## 2. Clone Ark+ repo, patch it, and install dependencies
The original repo pins 2021-era package versions. `timm==0.5.4` and `albumentations==0.4.3` are kept exact — the repo's README warns newer timm versions changed the Swin Transformer architecture enough to break pretrained-weight loading, and the dataloader uses transform names albumentations removed later. We patch one `np.bool` alias (removed in numpy>=1.24) instead of downgrading numpy globally, to avoid conflicting with Colab's preinstalled stack.

Patches applied here (all verified locally first):
- `models.py` imports `UperNet_swin` from a module that isn't actually included in this repo folder — a dead, unused import that crashes on load. Removed.
- `dataloader.py`'s `CheXpert` dataset always reads all 14 label columns. We add an optional `class_indices` argument so it can select just the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) in the right order — this is what powers the `NUM_CLASSES=5` option in cell 12's toggle. With `NUM_CLASSES=14` (the current default), `class_indices` is left `None` and the dataset falls back to its original all-14-columns behavior.
- Three data-driven improvements (patches 10-12), motivated by gaps observed in the initial baseline run and recent chest X-ray literature — each independently toggleable in cell 14:
  - **Per-pathology uncertain-label strategy** (patch 10): `dataloader.py`'s `uncertain_label` can now be a list (one strategy per class) instead of only a single global string — the original CheXpert paper's own U-Ones/U-Zeros ablation found the best strategy is pathology-dependent.
  - **Class-weighted loss** (patches 11/12): a `pos_weight` on the BCE loss, computed from the real training-set class balance, to counter the imbalance behind low-AP classes like Consolidation.
  - **Co-occurrence-aware regularization** (patches 11/12): a small penalty (`utils.CoOccurrenceBCELoss`) that nudges the classifier head's weight vectors for commonly co-occurring pathologies to point in more similar directions — a simplified, batch-size-independent take on hierarchical/dependency-aware losses in recent literature (e.g. HBCE, arXiv:2502.03591).

This cell is safe to re-run (e.g. after a kernel restart) — the clone and patches are skipped/no-op if already present.

In [ ]:
import os, sys

# Install the same dependencies the training code needs (this doesn't change: copying our own
# code below doesn't install third-party packages).
!{sys.executable} -m pip install -q --break-system-packages timm==0.5.4 albumentations==0.4.3 opencv-python-headless scikit-image scikit-learn \
    simpleitk scipy pydicom yacs einops transformers pretrainedmodels imgaug pillow tqdm

# imgaug (a transitive dep of albumentations==0.4.3, old and unmaintained) has a np.sctypes
# reference (removed in numpy 2.0) in parameters.py, plus a dead `from scipy import misc` import
# (removed in modern scipy) in the same file. Both crash on *import*, so we can't `import imgaug`
# first to locate the file -- that import is exactly what crashes. Using importlib.util.find_spec
# instead, which locates the module without executing it.
import importlib.util, os as _os
_ia_dir = _os.path.dirname(importlib.util.find_spec('imgaug').origin)
_ia_params = _os.path.join(_ia_dir, 'parameters.py')
_s = open(_ia_params).read()
_before = _s
_s = _s.replace('np.sctypes["float"]', '[np.float16, np.float32, np.float64]')
_s = _s.replace('from scipy import misc\n', '')
if _s != _before:
    open(_ia_params, 'w').write(_s)
    print('imgaug patched:', _ia_params)
else:
    print('imgaug already patched (or patterns not found), skipping:', _ia_params)
import imgaug
print('imgaug import OK, version', imgaug.__version__)

# The actual CheXpert training code (dataloader/trainer/engine/models) is pulled straight from
# Drive instead of re-deriving ~15 sed/python patches against a live jlianglab/Ark clone every
# run. This copy already has everything: base compatibility fixes, all 6 models wired in, and
# the full frozen protocol (320px support for every architecture, patient-level held-out test
# split, stratified ablation subset, ECE metric, configurable AMP dtype, per-trial seeding) --
# kept in sync with the canonical copy at github.com/fei7fei/chexpert-baselines. Every notebook
# reads from this exact same folder, so the underlying code is guaranteed byte-identical across
# all 6 models -- required for the frozen-protocol comparison to be valid.
from google.colab import drive
drive.mount('/content/drive')

_DRIVE_CODE_SRC = '/content/drive/MyDrive/Project/models/Repos/ark_plus_repo/Ark_Plus/Finetuning'
if not os.path.exists('/content/chexpert_code'):
    import shutil
    shutil.copytree(_DRIVE_CODE_SRC, '/content/chexpert_code')
%cd /content/chexpert_code
print('Loaded pre-patched CheXpert training code (frozen protocol v1) from Drive.')

In [ ]:
import torch, timm
print('torch', torch.__version__, 'cuda:', torch.cuda.is_available(), torch.cuda.get_device_name(0))
print('timm', timm.__version__)
from timm.models.swin_transformer import SwinTransformer
m = SwinTransformer(num_classes=5, img_size=768, patch_size=4, window_size=12, embed_dim=192,
                     depths=(2,2,18,2), num_heads=(6,12,24,48))
print('Swin-L (768, window12) built OK, params:', sum(p.numel() for p in m.parameters()))
del m

## 3. Mount Google Drive for checkpoint persistence
`/content` is ephemeral — if the runtime disconnects or gets recycled (e.g. Colab Compute Units running out), everything in it is gone, checkpoints included. This mounts Drive and points training's save location there instead, so progress survives even a full runtime loss. Re-running cell 14 after a restart will then automatically resume from the last saved epoch (via `args.resume`) instead of starting over. Mounted first (before the checkpoint step below) so that step can check Drive for an already-uploaded Ark+ checkpoint instead of always requiring the upload dialog.

In [ ]:
from google.colab import drive
import os
drive.mount('/content/drive')

DRIVE_CKPT_DIR = '/content/drive/MyDrive/ark_plus_chexpert_checkpoints'
os.makedirs(DRIVE_CKPT_DIR, exist_ok=True)
print('Checkpoints and results will be saved to:', DRIVE_CKPT_DIR)

## 3b. Ark+ pretrained checkpoint
`USE_ARK_WEIGHTS = True` (set below) uses the real Ark+ checkpoint (`Ark6_swinLarge768_ep50.pth.tar`, on Drive as of 2026-08-03, verified: backbone loads clean, only window-geometry buffers + head differ). Set to `False` to train Swin-L from random init instead (the SwinL control notebook's config).

Checks Drive first (`DRIVE_CKPT_DIR/Ark6_swinLarge768_ep50.pth.tar`) before falling back to the upload dialog — upload once, and it's there for every future session without re-uploading. Also more robust than relying on the upload dialog alone: `files.upload()` is the same class of browser-dependent Colab widget (`google.colab.userdata`) that silently failed earlier this session when connected through the VS Code↔Colab bridge rather than the browser.

In [ ]:
import os, shutil

USE_ARK_WEIGHTS = True  # real Ark+ checkpoint verified 2026-08-03 -- backbone loads clean, only window-geometry buffers + head differ, as expected

CKPT_FILENAME = 'Ark6_swinLarge768_ep50.pth.tar'
DRIVE_CKPT_PATH = os.path.join(DRIVE_CKPT_DIR, CKPT_FILENAME)
CKPT_PATH = f'/content/chexpert_code/{CKPT_FILENAME}'

if not USE_ARK_WEIGHTS:
    CKPT_PATH = None
    print('No checkpoint — will train Swin-L from random init.')
elif os.path.exists(DRIVE_CKPT_PATH):
    shutil.copy(DRIVE_CKPT_PATH, CKPT_PATH)
    print('Found checkpoint on Drive, using it directly:', CKPT_PATH)
else:
    print('Checkpoint not found on Drive -- falling back to upload dialog.')
    print('(If this hangs or fails silently through the VS Code<->Colab bridge, try running')
    print('this same cell from browser Colab instead, then rerun it here once uploaded.)')
    from google.colab import files
    uploaded = files.upload()  # select Ark6_swinLarge768_ep50.pth.tar
    ckpt_name = list(uploaded.keys())[0]
    os.rename(ckpt_name, CKPT_PATH)
    shutil.copy(CKPT_PATH, DRIVE_CKPT_PATH)  # so future sessions skip the upload entirely
    print('Uploaded and cached to Drive for next time:', DRIVE_CKPT_PATH)
    print('Using checkpoint:', CKPT_PATH)

## 4. Run finetuning on the CheXpert benchmark
Trains Swin-L per the `NUM_CLASSES` toggle set in the cell below, reporting AUROC/MCC/AP/F1 on the 5 competition classes (Atelectasis, Cardiomegaly, Consolidation, Edema, Pleural Effusion) each epoch either way. Adjust `BATCH_SIZE`/`EPOCHS`/`PATIENCE` there for your GPU and time budget.

Three toggles just below `NUM_CLASSES` control the data-driven improvements from section 2 (patches 10-12) — all default `True`, set any to `False` to A/B against the plain baseline:
- `USE_PER_CLASS_UNCERTAIN_LABEL` — per-pathology uncertain-label strategy instead of one global `"LSR-Ones"`.
- `USE_CLASS_WEIGHTED_LOSS` — `pos_weight` on the BCE loss, computed from the actual training-set class balance.
- `USE_COOCCURRENCE_LOSS` — co-occurrence-aware regularization on the classifier head's weights.

`args.exp_name` (and so the checkpoint directory) encodes which of these are active, so flipping a toggle starts a fresh run instead of accidentally resuming into a mismatched one.

In [ ]:
%cd /content/chexpert_code
import sys, os
sys.path.insert(0, '.')

_old_argv = sys.argv
sys.argv = ['main_classification.py']
from main_classification import get_args_parser
args = get_args_parser()
sys.argv = _old_argv

# ===== Frozen protocol v1 -- identical across all 6 models, fix this before running anything =====
INPUT_SIZE = 320                    # native CheXpert-small short side -- avoids gratuitous up/downsampling
UNCERTAIN_LABEL = "Zeros"           # U-Zeros for main results; flip to "Ones" for the appendix sensitivity check
EPOCHS = 30
SEEDS_SCREENING = [13]              # 1 seed: cheap ranking pass on the 20% ablation subset
SEEDS_HEADLINE = [13, 42]     # 3 seeds: confirmation pass on the full training pool

args.model_name = "swin_large"
args.init = "ark" if USE_ARK_WEIGHTS else "random"
args.pretrained_weights = CKPT_PATH
args.scale_up = True  # window_size changes 12 (768px pretrain) -> 5 (320px protocol); needed for the shape-mismatch filter in load_pretrained_weights
args.input_size = INPUT_SIZE
args.img_size = INPUT_SIZE
args.num_class = 14   # train the head on all 14 CheXpert labels; report only on the 5 competition
                       # classes via test_diseases below (original Ark+ protocol convention)
args.data_set = "CheXpert"
args.data_dir = "/content/data"
args.key = "teacher"
args.batch_size = 64   # "largest that fits" per protocol -- reduce if you hit OOM at 320px,
                                  # or wire in gradient checkpointing (not done here)
args.epochs = EPOCHS
args.patience = EPOCHS           # no early stopping -- every model trains the full 50 epochs so run
                                  # lengths stay comparable across the whole comparison
args.opt = "adamw"
args.lr = 5e-5
args.weight_decay = 0.05
args.sched = "cosine"
args.warmup_epochs = 1
args.cooldown_epochs = 0
args.uncertain_label = UNCERTAIN_LABEL
args.test_every_epoch = True
args.save_every_epoch_checkpoint = True
args.amp_dtype = "fp16"
args.device = "cuda"
args.resume = True
args.workers = 8  # bumped from 2: a rock-steady per-batch time with zero jitter is the signature of a CPU-bound dataloader, not a slow GPU

diseases_14 = ['No Finding', 'Enlarged Cardiomediastinum', 'Cardiomegaly', 'Lung Opacity',
               'Lung Lesion', 'Edema', 'Consolidation', 'Pneumonia', 'Atelectasis', 'Pneumothorax',
               'Pleural Effusion', 'Pleural Other', 'Fracture', 'Support Devices']
competition_5 = ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Pleural Effusion']
test_diseases_5 = [diseases_14.index(c) for c in competition_5]

from dataloader import CheXpert, build_transform_classification, build_protocol_splits, filter_csv_by_patients
from engine import classification_engine

# Patient-level held-out test (carved from train) + stratified 20% ablation subset -- computed
# once and cached to a *shared* Drive folder (not this model's own checkpoint folder) so every
# one of the 6 models' notebooks reads the exact same patient partition, in whatever order they
# happen to run. That's what makes the cross-model comparison valid rather than 6 separate
# anecdotes on 6 different splits.
PROTOCOL_SHARED_DIR = '/content/drive/MyDrive/chexpert_protocol_v1'
os.makedirs(PROTOCOL_SHARED_DIR, exist_ok=True)

train_csv = os.path.join(args.data_dir, "train.csv")
splits = build_protocol_splits(train_csv, competition_5, cache_path=os.path.join(PROTOCOL_SHARED_DIR, "protocol_split.csv"))

def _cached_filter(name, patient_set):
    p = os.path.join(PROTOCOL_SHARED_DIR, f"{name}.csv")
    if not os.path.exists(p):
        filter_csv_by_patients(train_csv, patient_set, p)
    return p

train_pool_csv = _cached_filter("train_pool", splits["train_pool"])
held_out_test_csv = _cached_filter("held_out_test", splits["held_out_test"])
ablation_subset_csv = _cached_filter("ablation_subset", splits["ablation_subset"])
official_val_csv = os.path.join(args.data_dir, "valid.csv")

args.train_list = train_pool_csv
args.val_list = official_val_csv     # the official 234-study set -- used for per-epoch threshold
                                      # tuning/checkpoint selection only, never for final reporting
args.test_list = held_out_test_csv   # final reporting -- held out from train, never seen in training

def make_ds(mode, file_path):
    return CheXpert(images_path=args.data_dir, file_path=file_path,
        augment=build_transform_classification(normalize=args.normalization, mode=mode, crop_size=args.input_size, resize=args.img_size),
        uncertain_label=args.uncertain_label, unknown_label=args.unknown_label)

dataset_val = make_ds("valid", args.val_list)
dataset_test = make_ds("test", args.test_list)
print(f"val (official, threshold tuning): {len(dataset_val)}  |  test (held-out, final report): {len(dataset_test)}")

model_path = os.path.join(DRIVE_CKPT_DIR, "Models/Classification", args.data_set)
output_path = os.path.join(DRIVE_CKPT_DIR, "Outputs/Classification", args.data_set)

# ===== Tier 1: screening -- cheap, single seed, ranks variants on the 20% subset =====
print("\n=== TIER 1: screening (1 seed, 20% stratified subset) ===")
args.exp_name = "_protocol1_arkplus_screening"
args.num_trial = len(SEEDS_SCREENING)
args.start_index = 0
args.seed_list = SEEDS_SCREENING
dataset_train_screen = make_ds("train", ablation_subset_csv)
print(f"screening train size: {len(dataset_train_screen)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_screen, dataset_val, dataset_test, test_diseases=test_diseases_5)

# ===== Tier 2: headline -- 3 seeds, full training data. Report both tiers; the two-tier split =====
# ===== is itself part of what makes ~20 runs comparable rather than 20 anecdotes. =====
print("\n=== TIER 2: headline (3 seeds, full training pool) ===")
args.exp_name = "_protocol1_arkplus_headline"
args.num_trial = len(SEEDS_HEADLINE)
args.start_index = 0
args.seed_list = SEEDS_HEADLINE
dataset_train_full = make_ds("train", train_pool_csv)
print(f"headline train size: {len(dataset_train_full)}")
classification_engine(args, model_path, output_path, diseases_14, dataset_train_full, dataset_val, dataset_test, test_diseases=test_diseases_5)

## New: class-weighted BCE quick test (proof-of-viability, screening tier only)

Does **not** modify the baseline training cell above. Computes `pos_weight` per class from the
real positive/negative counts in the screening training subset, then reruns the exact same
protocol (30 epochs, 1 screening seed) under a separate `exp_name` -- so it writes to its own
folder on Drive and never overwrites the plain-baseline checkpoint/results.

In [ ]:
## Class-weighted BCE quick test -- proof-of-viability for planned intervention (i) in the
## SPIE abstract. Targets the Atelectasis/Consolidation AP/F1 gap found in the Ark+ headline
## result. Screening tier only (1 seed, 20% subset, still 30 epochs -- matches the protocol's
## epoch count, only the seed/tier scope is cut for speed). Requires cell 14 above to have run
## at least once in this session (needs args, model_path, output_path, dataset_val, dataset_test,
## dataset_train_screen, ablation_subset_csv, diseases_14, test_diseases_5, SEEDS_SCREENING).

import copy
import shutil
import pandas as pd
import torch

_CW_EXP = "_protocol1_arkplus_cw_screening"

# Clean slate: delete any previous attempt at this exact experiment before training, so a
# fixed/changed pos_weight (or any other edit to this cell) can never silently resume from a
# checkpoint trained under different conditions -- args.resume=True would otherwise pick up
# whatever's on Drive under this exp_name, no matter how it got there. Hardcoded to _CW_EXP
# only -- never touches the plain baseline (_protocol1_arkplus_screening) or headline folders.
_cw_dir = os.path.join(model_path, _CW_EXP)
_cw_results_txt = os.path.join(output_path, _CW_EXP + "_results.txt")
if os.path.exists(_cw_dir):
    shutil.rmtree(_cw_dir)
    print(f"Deleted previous {_CW_EXP} checkpoint/logs at {_cw_dir} -- starting clean.")
if os.path.exists(_cw_results_txt):
    os.remove(_cw_results_txt)
    print(f"Deleted previous {_CW_EXP} results file at {_cw_results_txt}.")

_train_df = pd.read_csv(ablation_subset_csv)[diseases_14]
_n = len(_train_df)
_pos_count = (_train_df == 1.0).sum()
# Everything that ISN'T a definite 1 -- definite 0, uncertain (-1), and blank -- becomes a
# training-time negative under this protocol's uncertain_label="Zeros"/unknown_label=0, so the
# effective negative count is (n - pos_count), not just rows explicitly marked 0. Getting this
# wrong matters a lot here: Atelectasis alone has ~25x more uncertain-labeled rows than
# definite-negative rows in the full CheXpert train set, so counting only definite 0s produces
# a pos_weight two orders of magnitude too low -- i.e. down-weighting the exact class this test
# is meant to help.
pos_weight = torch.tensor(((_n - _pos_count) / _pos_count.clip(lower=1)).values, dtype=torch.float32)

print("pos_weight per class (screening subset, matches training-time Zeros/unknown=0 semantics):")
for _d, _w in zip(diseases_14, pos_weight.tolist()):
    print(f"  {_d:<28s} {_w:6.2f}")

cw_args = copy.deepcopy(args)
cw_args.exp_name = _CW_EXP  # separate from "_protocol1_arkplus_screening" -- never overwrites the plain baseline
cw_args.pos_weight = pos_weight
cw_args.num_trial = len(SEEDS_SCREENING)
cw_args.start_index = 0
cw_args.seed_list = SEEDS_SCREENING

print(f"\n=== Class-weighted BCE screening run: {cw_args.exp_name} ===")
classification_engine(cw_args, model_path, output_path, diseases_14, dataset_train_screen, dataset_val, dataset_test, test_diseases=test_diseases_5)

In [ ]:
## Compare class-weighted vs. plain-BCE screening results, focused on Atelectasis/Consolidation
## (the classes this intervention targets). Reads cached per-class predictions for both
## experiments from Drive -- the plain-baseline ones from cell 14's screening tier (this notebook
## or the original baseline/gaps notebook, since all three write to the same DRIVE_CKPT_DIR), the
## class-weighted ones from the cell above. 1-seed screening signal only -- directional, not a
## headline-tier confirmed result (Experiment Plan v3 SS6.3 on not treating n=1 as an error bar).

import os
import pandas as pd
from sklearn.metrics import average_precision_score, f1_score
from utils import read_from_csv

def _load_run(exp_name, run_idx=0):
    exp_dir = os.path.join(model_path, exp_name)
    run = f"{exp_name}_run_{run_idx}"
    pred_csv = os.path.join(exp_dir, run + ".csv")
    gt_csv = os.path.join(exp_dir, "gt.csv")
    if not (os.path.exists(pred_csv) and os.path.exists(gt_csv)):
        print(f"Missing cached predictions for {run} -- run its training cell first.")
        return None, None
    return read_from_csv(gt_csv), read_from_csv(pred_csv)

y_plain, p_plain = _load_run("_protocol1_arkplus_screening")
y_cw, p_cw = _load_run("_protocol1_arkplus_cw_screening")

if y_plain is not None and y_cw is not None:
    rows = []
    for i, cls in enumerate(competition_5):
        ap_plain = average_precision_score(y_plain[:, i], p_plain[:, i])
        ap_cw = average_precision_score(y_cw[:, i], p_cw[:, i])
        f1_plain = f1_score(y_plain[:, i], (p_plain[:, i] > 0.5).astype(int))
        f1_cw = f1_score(y_cw[:, i], (p_cw[:, i] > 0.5).astype(int))
        rows.append({
            "class": cls,
            "AP_plain": round(ap_plain, 4), "AP_cw": round(ap_cw, 4), "AP_delta": round(ap_cw - ap_plain, 4),
            "F1_plain": round(f1_plain, 4), "F1_cw": round(f1_cw, 4), "F1_delta": round(f1_cw - f1_plain, 4),
        })
    cmp_df = pd.DataFrame(rows)
    print("Screening tier, 1 seed each -- plain BCE vs. class-weighted BCE, all 5 competition classes:")
    display(cmp_df)
    print("\nAtelectasis/Consolidation are the targeted classes -- the ones this intervention is meant to move.")
    print("1-seed screening signal, not a headline-tier confirmed result -- treat directionally.")
else:
    print("Run both screening experiments (plain baseline in cell 14, class-weighted above) before comparing.")